# 🐴 競馬予測モデル学習（データリーク対策版）

**使い方：**
1. セル①でライブラリインストール
2. セル②でSupabase接続・データ取得
3. セル③で時系列分割
4. セル④でデータリーク対策（特徴量を過去データのみで再計算）
5. セル⑤でLightGBM学習
6. セル⑥で評価・特徴量重要度
7. セル⑦でモデル保存

In [ ]:
# ① ライブラリインストール
!pip install supabase lightgbm scikit-learn matplotlib seaborn -q

In [ ]:
# ② Supabase接続・データ取得
import pandas as pd
import numpy as np
from supabase import create_client

SUPABASE_URL = 'https://infypumigexmpdmijhnx.supabase.co'
SUPABASE_KEY = 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...'  # service_role keyを貼り付け

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Supabase接続OK')

# race_entriesとracesを結合して取得
all_data = []
offset = 0
while True:
    res = supabase.table('race_entries').select(
        'entry_id, race_id, horse_id, jockey_id, trainer_id,'
        'post_position, horse_number, finish_position,'
        'odds, popularity, weight, weight_diff'
    ).not_.is_('finish_position', 'null').range(offset, offset+999).execute()
    if not res.data: break
    all_data.extend(res.data)
    offset += 1000
    print(f'取得済み: {len(all_data)}件', end='\r')

df_entries = pd.DataFrame(all_data)

# racesテーブルを取得
all_races = []
offset = 0
while True:
    res = supabase.table('races').select(
        'race_id, race_date, venue_code, distance, surface,'
        'track_condition, weather, class'
    ).range(offset, offset+999).execute()
    if not res.data: break
    all_races.extend(res.data)
    offset += 1000

df_races = pd.DataFrame(all_races)
df_races['race_date'] = pd.to_datetime(df_races['race_date'])

# horse_pedigreesを取得
all_peds = []
offset = 0
while True:
    res = supabase.table('horse_pedigrees').select('horse_id, father, mother_father').range(offset, offset+999).execute()
    if not res.data: break
    all_peds.extend(res.data)
    offset += 1000

df_peds = pd.DataFrame(all_peds)

# 結合
df = df_entries.merge(df_races, on='race_id', how='inner')
df = df.merge(df_peds, on='horse_id', how='left')

# 出走頭数
df['entry_count'] = df.groupby('race_id')['race_id'].transform('count')

# ターゲット変数
df['target'] = (df['finish_position'] <= 3).astype(int)

print(f'\n✅ データ取得完了: {len(df):,}件')
print(f'期間: {df.race_date.min().date()} 〜 {df.race_date.max().date()}')

In [ ]:
# ③ 時系列分割
# 学習：〜2024年末、検証：2025年前半、テスト：2025年後半〜
TRAIN_END = '2024-12-31'
VALID_END = '2025-06-30'

train_mask = df['race_date'] <= TRAIN_END
valid_mask = (df['race_date'] > TRAIN_END) & (df['race_date'] <= VALID_END)
test_mask  = df['race_date'] > VALID_END

print(f'学習: {train_mask.sum():,}件 ({df[train_mask].race_date.min().date()} 〜 {df[train_mask].race_date.max().date()})')
print(f'検証: {valid_mask.sum():,}件')
print(f'テスト: {test_mask.sum():,}件')

In [ ]:
# ④ データリーク対策：特徴量を学習データのみで計算
# ポイント：テスト・検証データの特徴量は学習データの統計値でのみ計算する

def calc_fukusho_rate(df_train, group_cols):
    """学習データから複勝率を計算して辞書で返す"""
    stats = df_train.groupby(group_cols)['target'].agg(['mean', 'count'])
    stats.columns = ['fukusho_rate', 'count']
    # サンプル数10件未満はNaNにする
    stats.loc[stats['count'] < 10, 'fukusho_rate'] = np.nan
    return stats['fukusho_rate'].to_dict()

train_df = df[train_mask].copy()

# 騎手複勝率（学習データのみで計算）
print('騎手統計を計算中...')
jockey_stats = calc_fukusho_rate(train_df, ['jockey_id', 'venue_code', 'surface'])
jockey_stats_overall = calc_fukusho_rate(train_df, ['jockey_id'])

# 調教師複勝率
print('調教師統計を計算中...')
trainer_stats = calc_fukusho_rate(train_df, ['trainer_id', 'surface'])

# 父産駒複勝率
print('父統計を計算中...')
father_stats = calc_fukusho_rate(train_df, ['father', 'surface'])

# 母父複勝率
print('母父統計を計算中...')
mf_stats = calc_fukusho_rate(train_df, ['mother_father', 'surface'])

# ニックス（父×母父）複勝率
print('ニックス統計を計算中...')
nick_stats = calc_fukusho_rate(train_df, ['father', 'mother_father', 'surface'])

# 馬の過去成績（対象レースより前のデータのみ）
print('馬の過去成績を計算中...')
df_sorted = df.sort_values(['horse_id', 'race_date'])
horse_stats = []
for idx, row in df_sorted.iterrows():
    past = df_sorted[(df_sorted['horse_id'] == row['horse_id']) &
                     (df_sorted['race_date'] < row['race_date'])]
    past_3 = past.tail(3)
    past_5 = past.tail(5)
    horse_stats.append({
        'entry_id': row['entry_id'],
        'avg_finish_3': past_3['finish_position'].mean() if len(past_3) > 0 else np.nan,
        'fukusho_rate_3': (past_3['target'].mean()) if len(past_3) > 0 else np.nan,
        'avg_finish_5': past_5['finish_position'].mean() if len(past_5) > 0 else np.nan,
        'fukusho_rate_5': (past_5['target'].mean()) if len(past_5) > 0 else np.nan,
        'last_finish': past.iloc[-1]['finish_position'] if len(past) > 0 else np.nan,
        'rest_days': (row['race_date'] - past.iloc[-1]['race_date']).days if len(past) > 0 else np.nan,
        'distance_diff': (row['distance'] - past.iloc[-1]['distance']) if len(past) > 0 else np.nan,
    })

df_horse_stats = pd.DataFrame(horse_stats)
df = df.merge(df_horse_stats, on='entry_id', how='left')
print('✅ 馬の過去成績計算完了')

In [ ]:
# ④-2 特徴量を全データに適用
def apply_stats(df, col_name, stats_dict, key_cols):
    df[col_name] = df[key_cols].apply(
        lambda x: stats_dict.get(tuple(x) if len(key_cols) > 1 else x.iloc[0], np.nan), axis=1
    )

# 騎手（会場×芝ダ）、なければ全体平均でフォールバック
df['jockey_fukusho_rate'] = df[['jockey_id', 'venue_code', 'surface']].apply(
    lambda x: jockey_stats.get((x['jockey_id'], x['venue_code'], x['surface']),
              jockey_stats_overall.get(x['jockey_id'], np.nan)), axis=1
)
df['trainer_fukusho_rate'] = df[['trainer_id', 'surface']].apply(
    lambda x: trainer_stats.get((x['trainer_id'], x['surface']), np.nan), axis=1
)
df['father_fukusho_rate'] = df[['father', 'surface']].apply(
    lambda x: father_stats.get((x['father'], x['surface']), np.nan), axis=1
)
df['mother_father_fukusho_rate'] = df[['mother_father', 'surface']].apply(
    lambda x: mf_stats.get((x['mother_father'], x['surface']), np.nan), axis=1
)
df['nick_fukusho_rate'] = df[['father', 'mother_father', 'surface']].apply(
    lambda x: nick_stats.get((x['father'], x['mother_father'], x['surface']), np.nan), axis=1
)

print('✅ 特徴量適用完了')

In [ ]:
# ⑤ カテゴリ変数エンコード & LightGBM学習
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

cat_cols = ['venue_code', 'surface', 'track_condition', 'weather', 'class']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].fillna('unknown').astype(str))

feature_cols = [
    # レース条件
    'venue_code_enc', 'distance', 'surface_enc', 'track_condition_enc',
    'weather_enc', 'post_position', 'entry_count', 'class_enc',
    # 馬の過去成績
    'avg_finish_3', 'avg_finish_5', 'fukusho_rate_3', 'fukusho_rate_5',
    'last_finish', 'rest_days', 'distance_diff', 'weight', 'weight_diff',
    # 騎手・調教師
    'jockey_fukusho_rate', 'trainer_fukusho_rate',
    # 血統
    'father_fukusho_rate', 'mother_father_fukusho_rate', 'nick_fukusho_rate',
    # オッズ・人気
    'odds', 'popularity',
    'popularity_ratio',
]

# popularity_ratio
df['popularity_ratio'] = df['popularity'] / df['entry_count']

X_train = df[train_mask][feature_cols]
y_train = df[train_mask]['target']
X_valid = df[valid_mask][feature_cols]
y_valid = df[valid_mask]['target']
X_test  = df[test_mask][feature_cols]
y_test  = df[test_mask]['target']

print(f'学習: {len(X_train):,}件 | 検証: {len(X_valid):,}件 | テスト: {len(X_test):,}件')

params = {
    'objective': 'binary', 'metric': 'auc',
    'num_leaves': 31, 'learning_rate': 0.05,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'min_child_samples': 20, 'class_weight': 'balanced',
    'verbose': -1,
}

model = lgb.LGBMClassifier(**params, n_estimators=1000)
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

if len(X_test) > 0:
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f'\n✅ テストAUC: {auc:.4f}')

In [ ]:
# ⑥ 特徴量重要度
import matplotlib.pyplot as plt

lgb.plot_importance(model, max_num_features=20, figsize=(10, 8))
plt.title('特徴量重要度')
plt.tight_layout()
plt.show()

In [ ]:
# ⑦ バックテスト（期待値フィルタ別回収率）
# 複勝オッズをpayoutsから取得
all_pays = []
offset = 0
while True:
    res = supabase.table('payouts').select('race_id, combination, payout').eq('bet_type', '複勝').range(offset, offset+999).execute()
    if not res.data: break
    all_pays.extend(res.data)
    offset += 1000

df_pay = pd.DataFrame(all_pays)
df_pay['horse_number'] = pd.to_numeric(df_pay['combination'], errors='coerce')
df_pay['fukusho_odds'] = df_pay['payout'] / 100

df_test = df[test_mask].copy()
df_test['proba'] = model.predict_proba(X_test)[:, 1]
df_test = df_test.merge(df_pay[['race_id', 'horse_number', 'fukusho_odds']], on=['race_id', 'horse_number'], how='left')
df_test['expected_value'] = df_test['proba'] * df_test['fukusho_odds'].fillna(0)

print('=' * 55)
print(f"{'閾値':>8} | {'購入件数':>8} | {'的中率':>8} | {'回収率':>8}")
print('-' * 55)
for threshold in [0.8, 1.0, 1.1, 1.2, 1.3, 1.5]:
    buy = df_test[df_test['expected_value'] > threshold]
    if len(buy) == 0: continue
    total_bet = len(buy) * 100
    total_ret = (buy[buy['target']==1]['fukusho_odds'] * 100).sum()
    recovery  = total_ret / total_bet * 100
    hit_rate  = buy['target'].mean() * 100
    print(f'{threshold:>8.1f} | {len(buy):>8,} | {hit_rate:>7.1f}% | {recovery:>7.1f}%')
print('=' * 55)

In [ ]:
# ⑧ モデル保存
import joblib
from datetime import datetime

date_str = datetime.now().strftime('%Y%m%d')
fname = f'keiba_model_{date_str}.pkl'
joblib.dump(model, fname)
print(f'✅ モデル保存: {fname}')